In [36]:
import requests
import time
import json, os

In [37]:
API = "https://api.stackexchange.com/2.3"
SITE = "bioinformatics.stackexchange.com"
API_KEY = "rl_ctU1teLi66r1cEfnbErDWgJdg"
HEADERS = {"User-Agent": "MetaGenomicAgent-QAScraper/1.0"}

In [3]:
def fetch_all_tags(pagesize=100):
    page = 1
    tags = []

    while True:
        url = f"{API}/tags?page={page}&pagesize={pagesize}&order=desc&sort=popular&site={SITE}"
        print("Fetching:", url)
        r = requests.get(url).json()

        if "items" not in r:
            break

        tags.extend([t["name"] for t in r["items"]])

        if not r.get("has_more"):
            break

        page += 1
        time.sleep(0.2)  

    return tags

In [4]:
tags = fetch_all_tags()
print("Total tags:", len(tags))
print(tags[:50])

Fetching: https://api.stackexchange.com/2.3/tags?page=1&pagesize=100&order=desc&sort=popular&site=bioinformatics.stackexchange.com
Fetching: https://api.stackexchange.com/2.3/tags?page=2&pagesize=100&order=desc&sort=popular&site=bioinformatics.stackexchange.com
Fetching: https://api.stackexchange.com/2.3/tags?page=3&pagesize=100&order=desc&sort=popular&site=bioinformatics.stackexchange.com
Fetching: https://api.stackexchange.com/2.3/tags?page=4&pagesize=100&order=desc&sort=popular&site=bioinformatics.stackexchange.com
Fetching: https://api.stackexchange.com/2.3/tags?page=5&pagesize=100&order=desc&sort=popular&site=bioinformatics.stackexchange.com
Fetching: https://api.stackexchange.com/2.3/tags?page=6&pagesize=100&order=desc&sort=popular&site=bioinformatics.stackexchange.com
Total tags: 528
['r', 'phylogenetics', 'rna-seq', 'python', 'ngs', 'scrnaseq', 'phylogeny', 'proteins', 'vcf', 'fasta', 'sam', 'genome', 'protein-structure', 'gene', 'seurat', 'bioconductor', 'sequence-annotation',

In [ ]:
with open("../04-output/raw/biostackexchange/tags.json", "w", encoding="utf-8") as f:
   json.dump(tags, f, indent=4, ensure_ascii=False)

In [9]:
####################

In [38]:
# --------------------------
# READ TAGS FILE
# --------------------------
with open("../04-output/raw/biostackexchange/tags_filtered.json", "r", encoding="utf-8") as f:
    TAGS = json.load(f)

print("Loaded tags:", len(TAGS))

Loaded tags: 74


In [39]:
# --------------------------
# UTILS
# --------------------------
def safe_json(resp):
    try:
        return resp.json()
    except Exception:
        print("❌ Not JSON, response was:")
        print(resp, "...")
        return None
    
    
# --------------------------
# FETCH ANSWERS
# --------------------------

# [old]
def fetch_answers(qid):
    url = (
        f"{API}/questions/{qid}/answers"
        f"?order=desc&sort=activity&site={SITE}&filter=withbody"
    )
    r = requests.get(url, headers=HEADERS).json()
    return r.get("items", [])

# [new: we optimize the number of api call to take only what matter]
def fetch_answers_2(qid):
    url = (
        f"{API}/answers/{qid}"
        f"?site={SITE}&filter=withbody"
    )
    r = requests.get(url, headers=HEADERS).json()
    return r.get("items", [])

# [even more optimized]
def fetch_answers_batch(ids):
    if not ids:
        return {}

    ids_str = ";".join(map(str, ids))
    page = 1
    results = {}
    
    while True:
        url = (
            f"{API}/answers/{ids_str}"
            f"?page={page}&pagesize=100"
            f"&site={SITE}&filter=withbody&key={API_KEY}"
        )

        resp = requests.get(url, headers=HEADERS)
        r = safe_json(resp)
        if not r:
            break

        if "backoff" in r:
            wait = r["backoff"]
            print(f"⛔ Backoff (answers): waiting {wait}s")
            time.sleep(wait)

        for a in r.get("items", []):
            results[a["answer_id"]] = a
        if not r.get("has_more"):
            break
        page += 1
        time.sleep(0.2)

    return results

In [40]:

# --------------------------
# FETCH QUESTIONS FOR ONE TAG
# --------------------------
def fetch_questions_for_tag(tag):
    page = 1
    results = []

    while True:
        url = (
            f"{API}/questions?page={page}&pagesize=100"
            f"&order=desc&sort=activity&tagged={tag}"
            f"&filter=withbody&site={SITE}&key={API_KEY}"
        )
        print(f"[{tag}] Page {page}")

        resp = requests.get(url, headers=HEADERS)
        r = safe_json(resp)
        
        # Respect "backoff"
        if r and "backoff" in r:
            wait = r["backoff"]
            print(f"⛔ Backoff requested: waiting {wait} seconds")
            time.sleep(wait)

        if not r or "items" not in r:
            break
        
        # ----------------------------
        # 1. Collect accepted answers
        # ----------------------------
        # questions = r["items"]
        questions = [q for q in r["items"] if "accepted_answer_id" in q]
        if not questions:
            pass
        
        accepted_map = {
            q["question_id"]: q["accepted_answer_id"]
            for q in questions if "accepted_answer_id" in q
        }
        answer_ids = list(accepted_map.values())

        # ----------------------------
        # 2. Batch fetch all answers
        # ----------------------------
        answers_dict = fetch_answers_batch(answer_ids)

        # ----------------------------
        # 3. Populate results
        # ----------------------------
        for q in questions:
            qid = q["question_id"]
            best_answer = None

            if qid in accepted_map:
                ans_id = accepted_map[qid]
                best_answer = answers_dict.get(ans_id)

            entry = {
                "url": q.get("link"),
                "title": q.get("title"),
                "question_text": q.get("body"),
                "question_created_at": q.get("creation_date"),

                "answers": [],
                "best_answer": best_answer,

                "tags_api": q.get("tags", []),
                "score": q.get("score", 0),
                "accepted": qid in accepted_map,
                # "owner_reputation": q["owner"]["reputation"]
                "owner_reputation": q.get("owner", {}).get("reputation")
            }

            results.append(entry)

        if not r.get("has_more"):
            break

        page += 1
        time.sleep(0.35)
        
        if "quota_remaining" in r:
            print("🔢 Quota remaining:", r["quota_remaining"])
            
    return results

In [41]:
OUTPUT_FILE = "../04-output/raw/biostackexchange/raw_qa_metagenomics.json"
CHECKPOINT_FILE = "../04-output/raw/biostackexchange/checkpoint_tag.txt"

# ------------------------------------
# Load last completed tag index
# ------------------------------------
def get_last_index():
    if os.path.exists(CHECKPOINT_FILE):
        try:
            return int(open(CHECKPOINT_FILE).read().strip())
        except:
            return -1
    return -1

# ------------------------------------
# Save current tag index as checkpoint
# ------------------------------------
def save_checkpoint(idx):
    with open(CHECKPOINT_FILE, "w") as f:
        f.write(str(idx))


# ------------------------------------
# MAIN COLLECTION (SIMPLE & CLEAN)
# ------------------------------------
all_flat = []

# If file exists, load previous results so far
if os.path.exists(OUTPUT_FILE):
    print("Resuming: loading existing data…")
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        try:
            all_flat = json.load(f)
        except json.JSONDecodeError:
            print("Warning: output file corrupted or empty, starting fresh.")

# Dedup by URL
existing_urls = {item["url"] for item in all_flat}

start_index = get_last_index() + 1
print(f"Starting from tag index {start_index}\n")

for idx, tag in enumerate(TAGS):
    if idx < start_index:
        continue 

    print(f"\nCollecting tag #{idx}: {tag}")
    data = fetch_questions_for_tag(tag)

    # Add only new ones
    new_items = [d for d in data if d["url"] not in existing_urls]
    all_flat.extend(new_items)

    for d in new_items:
        existing_urls.add(d["url"])

    print(f"✔ {tag}: {len(new_items)} new questions added")

    # ---- SAVE immediately after each tag ----
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(all_flat, f, indent=4, ensure_ascii=False)

    # ---- UPDATE CHECKPOINT ----
    save_checkpoint(idx)

print("\nDONE!")
print("TOTAL UNIQUE QUESTIONS:", len(all_flat))


Starting from tag index 0


[metagenome] Page 1
✔ metagenome: 35 new questions added

[microbial-genomics] Page 1
✔ microbial-genomics: 3 new questions added

[microbiome] Page 1
✔ microbiome: 0 new questions added

[taxonomy] Page 1
✔ taxonomy: 11 new questions added

[bacteria] Page 1
✔ bacteria: 10 new questions added

[16rrna] Page 1
✔ 16rrna: 2 new questions added

[radseq] Page 1
✔ radseq: 1 new questions added

[assembly] Page 1
🔢 Quota remaining: 9795
[assembly] Page 2
✔ assembly: 60 new questions added

[scaffold] Page 1
✔ scaffold: 3 new questions added

[k-mer] Page 1
✔ k-mer: 22 new questions added

[de-bruijn-graphs] Page 1
✔ de-bruijn-graphs: 0 new questions added

[reads] Page 1
✔ reads: 21 new questions added

[read-correction] Page 1
✔ read-correction: 3 new questions added

[long-reads] Page 1
✔ long-reads: 24 new questions added

[nanopore] Page 1
🔢 Quota remaining: 9778
[nanopore] Page 2
✔ nanopore: 40 new questions added

[minion] Page 1
✔ minion: 8 new questions a

In [ ]:
#all_flat[1]

In [29]:
import json

with open("../04-output/raw/biostackexchange/raw_qa_metagenomics_.json", "r", encoding="utf-8") as f:
    data = json.load(f)

seen = set()
duplicates = []

for item in data:
    url = item.get("url")
    if url in seen:
        duplicates.append(item)
    else:
        seen.add(url)

print("Number of duplicates:", len(duplicates))


Number of duplicates: 0
